# Demand Evidence & Willingness-to-Pay

**Component 2 of the BDA final project.** This notebook documents the *full process* by which we go
from a vague idea ("job-market data is valuable") to a defensible claim about demand and pricing.

It pulls live numbers from our own API (run `make demo` first), then layers on a competitor
benchmark and a survey design. Four converging evidence streams:
1. The scraped corpus itself proves employer demand and that the salary/skill signal is extractable.
2. A competitor comparison shows a real market with an unfilled cell.
3. A survey tests willingness-to-pay across segments.
4. Public corroboration (gov stats, forum threads).


In [ ]:
import urllib.request, json
import pandas as pd
API = 'http://localhost:8000'
def api(path):
    with urllib.request.urlopen(API+path, timeout=15) as r:
        return json.load(r)
print('API:', api('/health'))


## 1. The scraped corpus is the primary demand evidence

Quantify the corpus we built end-to-end.


In [ ]:
kpi = {k['metric']: k['value'] for k in api('/summary')}
pd.Series(kpi).to_frame('value')


In [ ]:
skills = pd.DataFrame(api('/skills?limit=15'))
skills[['skill','category','total_postings','latest_month','mom_pct']]


**Reading:** with only a sample corpus we already recover a ranked, categorized skill-demand table
with month-over-month momentum. At full scale (the live government feed alone is tens of thousands of
postings) this becomes a continuously-updated market map — exactly the artifact our customers lack.


## 2. The salary signal is real and extractable

Skill-level salary percentiles — the data competitors do not publish for Taiwan.


In [ ]:
sal = pd.DataFrame(api('/salary?region=ALL&seniority=ALL&limit=15'))
sal = sal.sort_values('p50', ascending=False)
sal[['skill','p25','p50','p75','sample_size']].head(15)


In [ ]:
# Quantified value: the salary premium of a 'hot' skill
d = {r['skill']: r['p50'] for r in api('/salary?region=ALL&seniority=ALL&limit=200')}
for s in ['Spark','Kubernetes','LLM']:
    if s in d:
        print(f'{s}: median NT${d[s]:,.0f}/month')


## 3. Demand trends over time


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
for s in ['Spark','Kubernetes','LLM','Go']:
    try: t = api(f'/skills/{s}/trend')
    except Exception: continue
    p = t['points']
    ax.plot([x['year_month'] for x in p], [x['postings'] for x in p], marker='o', label=s)
ax.set_title('Skill demand over time'); ax.set_ylabel('postings'); ax.legend(); plt.show()


## 4. Competitor benchmark (the market + the gap)

| Product | Taiwan-local | Skill-level | Real-time | Time-series | Notes |
|---|---|---|---|---|---|
| Levels.fyi | partial | no | crowd | no | English, big-tech-skewed, thin TW data |
| Glassdoor / LinkedIn Salary | partial | no | no | no | sparse for Taiwan |
| Robert Walters / Michael Page guides | yes | no | no (annual) | no | aggregate PDF, expensive |
| 104 薪資情報 / Yourator salary | yes | no | no | no | aggregate, not skill-level |
| **This project** | **yes** | **yes** | **yes** | **yes** | the unfilled cell |

No incumbent provides *real-time, skill-level, Taiwan-local, time-series* salary+demand data — our wedge.


## 5. Survey design & willingness-to-pay

A 10-minute survey (target n=30-50) across three segments, anchoring on concrete price points:

**Job seekers / students** (freemium funnel):
- *Would you pay for a personalized skill-gap + salary report for your target role?* NT$0 / 99 / 299
- *Which is more valuable: salary benchmark, or which skills to learn next?*

**Recruiters / HR** (data feed):
- *Would your team pay for a monthly skill-demand & comp-benchmark feed?* NT$/month: 0 / 2k / 5k / 10k
- *How many hours/month does your team spend manually compiling this today?*

**Bootcamps / universities / course providers** (B2B2X wedge):
- *Would you pay for skill-trend data to align curriculum?* annual license NT$: 0 / 50k / 150k

**WTP framing (time saved):** manually compiling a comparable skill+salary benchmark from 104 takes a
recruiter an estimated 6-10 hours/month; at NT$500/hr that is NT$3,000-5,000/month of labor a feed replaces.


## 6. Public corroboration

- Government employment statistics confirm sustained IT-sector hiring demand.
- PTT Tech_Job / Dcard 工程師 threads repeatedly ask "is skill X worth learning?" and "what salary should
  I ask for?" — the exact questions our product answers, evidence the pain is real and recurring.

## Conclusion

The corpus proves the signal exists and is extractable; competitors prove a market with an open cell;
the survey + time-saved framing give a defensible price. Together they justify a freemium B2C funnel
feeding paid B2B feeds and B2B2X curriculum licensing.
